# Route A — Pre-class checklist

Run this notebook **on every student laptop** the day before Session 1, in the same folder as the
lesson notebook. Each check prints `PASS`, `WARN` or `FAIL`. Fix every `FAIL` before class.

| # | Check | When |
|---|---|---|
| 1 | Python packages installed (`ultralytics`, `opencv-python`) | Once per laptop |
| 2 | Model file `yolo26n.pt` is in the lesson folder and loads | Once per laptop |
| 3 | Camera permission granted | Once per laptop |
| 4 | Laptop camera opens and gives a 640 × 480 picture | Once per laptop |
| 5 | Model speed is usable on this laptop | Once per laptop |
| 6 | Sample photos exist and give the expected detections | Before Session 1 |
| 7 | *(Robot only)* UGOT connects, and target sizes are calibrated | Before Session 8 |

The last cell prints a summary of all checks.

## Before you start — camera permission (check 3)

The camera will not open until the app running the notebook is allowed to use it.

- **macOS:** System Settings → Privacy & Security → Camera → turn on **Visual Studio Code**
  (or Terminal / Jupyter, whichever you use). Then quit VS Code fully with **Cmd+Q** and open it again.
  Closing the window is not enough.
- **Windows:** Settings → Privacy & security → Camera → turn on **Camera access** and
  **Let desktop apps access your camera**. Restart VS Code.
- **Linux:** the user must be able to read `/dev/video0` (usually by being in the `video` group).
  Log out and back in after changing groups.
- **Any laptop:** close Zoom, Teams, Photo Booth or any other app using the camera.

## Check 1 — Python packages

If `ultralytics` is missing, run the install cell below it **on the office network, not on class Wi-Fi**.
It also installs PyTorch, which is a large download (several hundred MB or more).

In [ ]:
import platform
import sys
import time
from pathlib import Path

report = {}

def record(check, status, detail=""):
    report[check] = (status, detail)
    print(f"{status:5}  {check}  {detail}")

print("Python", sys.version.split()[0], "on", platform.system(), platform.machine())

try:
    import cv2
    record("1a opencv", "PASS", cv2.__version__)
except ImportError:
    record("1a opencv", "FAIL", "run: %pip install opencv-python")

try:
    import ultralytics
    record("1b ultralytics", "PASS", ultralytics.__version__)
except ImportError:
    record("1b ultralytics", "FAIL", "run the install cell below, then restart the kernel")

In [ ]:
# Only run this if check 1 failed. Restart the kernel afterwards.
# %pip install ultralytics opencv-python

## Check 2 — Model file

Ultralytics downloads `yolo26n.pt` into the current folder the first time it is loaded.
To save class Wi-Fi, download it once, then copy the file into the lesson folder on every laptop
(USB stick or shared drive). The file is small (a few MB).

In [ ]:
from ultralytics import YOLO

model_file = Path("yolo26n.pt")
if model_file.exists():
    print("Found", model_file, f"({model_file.stat().st_size / 1e6:.1f} MB)")
else:
    print("yolo26n.pt is not in this folder yet - downloading it now.")

try:
    model = YOLO("yolo26n.pt")
    if len(model.names) == 80:
        record("2 model file", "PASS", "yolo26n.pt loads, knows 80 things")
    else:
        record("2 model file", "WARN", f"loaded, but knows {len(model.names)} things (expected 80)")
except Exception as error:
    model = None
    record("2 model file", "FAIL", str(error))

if not model_file.exists():
    print("Note: the model loaded but the file is not in this folder:", Path.cwd())

## Checks 3 and 4 — Laptop camera

This is the same `open_laptop_camera()` the lessons use. It picks the right camera driver for
macOS, Windows or Linux and asks for 640 × 480.

- `PASS` — the picture is 640 × 480.
- `WARN` — the camera works but ignored the size request. The lessons still work, because they read
  the centre from the picture, but boxes will look bigger or smaller than on other laptops.
- `FAIL` — no picture. Go back to the camera permission steps above. If the laptop has more than one
  camera, try changing `0` to `1` inside the function.

In [ ]:
import cv2
from IPython.display import display, Image


def open_laptop_camera():
    system = platform.system()
    if system == "Darwin":
        camera = cv2.VideoCapture(0, cv2.CAP_AVFOUNDATION)
    elif system == "Windows":
        camera = cv2.VideoCapture(0, cv2.CAP_DSHOW)
    else:
        camera = cv2.VideoCapture(0)

    camera.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    camera.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

    for attempt in range(50):
        ok, frame = camera.read()
        if ok and frame is not None:
            return camera, frame
        time.sleep(0.1)

    camera.release()
    raise RuntimeError("The camera did not start. Check camera permission, then restart VS Code.")


camera_frame = None
try:
    camera, camera_frame = open_laptop_camera()
    camera.release()
    record("3 camera permission", "PASS", "camera opened")

    height, width = camera_frame.shape[:2]
    if (width, height) == (640, 480):
        record("4 camera size", "PASS", "640 x 480")
    else:
        record("4 camera size", "WARN", f"{width} x {height} (asked for 640 x 480)")

    ok, jpg = cv2.imencode(".jpg", camera_frame)
    display(Image(jpg.tobytes()))
except RuntimeError as error:
    record("3 camera permission", "FAIL", str(error))
    record("4 camera size", "FAIL", "no picture")

## Check 5 — Model speed

Session 6 compares `imgsz=320` with `imgsz=640`. Under about 150 ms per picture at 320 is
comfortable for live video. Much slower than that and the robot in Session 8 will lag badly.

In [ ]:
if model is None:
    record("5 model speed", "FAIL", "model did not load")
else:
    import numpy as np
    test_picture = camera_frame if camera_frame is not None else np.zeros((480, 640, 3), np.uint8)

    timings = {}
    for size in (320, 640):
        model(test_picture, imgsz=size, verbose=False)          # warm-up run
        start = time.time()
        for i in range(5):
            model(test_picture, imgsz=size, verbose=False)
        timings[size] = (time.time() - start) / 5 * 1000
        print(f"imgsz={size}: {timings[size]:.0f} ms per picture")

    if timings[320] < 150:
        record("5 model speed", "PASS", f"{timings[320]:.0f} ms at 320")
    else:
        record("5 model speed", "WARN", f"{timings[320]:.0f} ms at 320 - live video will feel slow")

## Check 6 — Sample photos

The lessons need these photos in the same folder. What each one must give:

| Photo | Used in | Must give |
|---|---|---|
| `desk.jpg` | Sessions 1–3 | at least one detection (Session 3 takes the first box without checking) |
| `messy_desk.jpg` | Sessions 4–6 | at least one `bottle` and one `cup`, with at least one bottle above 0.5 |
| `blank_wall.jpg` | Session 4 | **zero** detections, or the planted crash will not happen |
| `my_photo.jpg` | Sessions 1–2 | optional — students replace it with their own photo |

If a photo fails, take a new one. For `messy_desk.jpg`, use a plain bottle and mug, well lit, filling a
good part of the picture.

In [ ]:
def detections(filename):
    picture = cv2.imread(filename)
    if picture is None:
        return None
    results = model(picture, verbose=False)
    found = []
    for box in results[0].boxes:
        found.append((model.names[int(box.cls[0])], round(float(box.conf[0]), 2)))
    return found


if model is None:
    record("6 sample photos", "FAIL", "model did not load")
else:
    desk = detections("desk.jpg")
    if desk is None:
        record("6a desk.jpg", "FAIL", "file missing or unreadable")
    elif len(desk) == 0:
        record("6a desk.jpg", "FAIL", "no detections - Session 3 will crash")
    else:
        record("6a desk.jpg", "PASS", str(desk))

    messy = detections("messy_desk.jpg")
    if messy is None:
        record("6b messy_desk.jpg", "FAIL", "file missing or unreadable")
    else:
        names = [name for name, conf in messy]
        sure_bottle = any(name == "bottle" and conf > 0.5 for name, conf in messy)
        if names.count("bottle") > 0 and names.count("cup") > 0 and sure_bottle:
            record("6b messy_desk.jpg", "PASS", str(messy))
        else:
            record("6b messy_desk.jpg", "FAIL",
                   f"needs a bottle above 0.5 and a cup - got {messy}")

    wall = detections("blank_wall.jpg")
    if wall is None:
        record("6c blank_wall.jpg", "FAIL", "file missing or unreadable")
    elif len(wall) == 0:
        record("6c blank_wall.jpg", "PASS", "zero detections")
    else:
        record("6c blank_wall.jpg", "FAIL", f"should find nothing - got {wall}")

    mine = detections("my_photo.jpg")
    if mine is None:
        record("6d my_photo.jpg", "WARN", "not there yet - students add it in Session 1")
    else:
        record("6d my_photo.jpg", "PASS", str(mine))

## Check 7 — Robot (only before Session 8)

Skip this section on laptops that will not drive a robot.

**Calibrating `target_sizes`:** put one object where the robot should stop (about a hand's length
in front of it), in the middle of the picture. Run the cell. Copy the width into `target_sizes` in the
lesson notebook. Repeat for each item in `lost_items`. The numbers only need to be done once per
robot model and camera, not once per laptop.

In [ ]:
RUN_ROBOT_CHECK = False           # change to True when a robot is on and connected
ROBOT_IP = "192.168.1.___"        # the IP address shown on the robot
CALIBRATE = ["bottle", "cup", "book"]

if RUN_ROBOT_CHECK:
    import numpy as np
    from ugot import ugot

    got = ugot.UGOT()
    got.initialize(ROBOT_IP)
    got.open_camera()

    robot_frame = None
    for attempt in range(50):
        raw = got.read_camera_data()
        if raw is not None:
            robot_frame = cv2.imdecode(np.frombuffer(raw, np.uint8), cv2.IMREAD_COLOR)
            if robot_frame is not None:
                break
        time.sleep(0.1)

    if robot_frame is None:
        record("7a robot camera", "FAIL", "no picture from the robot")
    else:
        height, width = robot_frame.shape[:2]
        record("7a robot camera", "PASS", f"{width} x {height}")

        results = model(robot_frame, imgsz=320, verbose=False)
        for box in results[0].boxes:
            name = model.names[int(box.cls[0])]
            conf = float(box.conf[0])
            if CALIBRATE.count(name) > 0 and conf > 0.5:
                x1, y1, x2, y2 = [int(v) for v in box.xyxy[0]]
                print(f"{name}: width {x2 - x1} px, bottom edge at y={y2} of {height}")
                if y2 > height - 5:
                    print("   the box touches the bottom of the picture - use a smaller target width")

        ok, jpg = cv2.imencode(".jpg", robot_frame)
        display(Image(jpg.tobytes()))
else:
    print("Robot check skipped.")

## Summary

In [ ]:
fails = [k for k, (status, detail) in report.items() if status == "FAIL"]
warns = [k for k, (status, detail) in report.items() if status == "WARN"]

print("Laptop:", platform.node(), "|", platform.system())
for check, (status, detail) in sorted(report.items()):
    print(f"  {status:5} {check}")

print()
if fails:
    print("NOT READY - fix:", ", ".join(fails))
elif warns:
    print("READY, with warnings:", ", ".join(warns))
else:
    print("READY for Route A.")